```markdown
1.2 Self-attention from first principles on a tiny example (NumPy only).
我们使用 T=3 个 tokens, d_model=4, d_k=d_v=4, 单头注意力。
这个脚本打印中间张量，以便追踪数学计算过程。

> 单头且 d_k=d_v=d_model=4：投影前后维度一致，便于理解「Q/K/V 与输入同维」。

维度总结 (单头注意力)
--------------------------------
X:          (B=1, T=3, d_model=4)  - 输入序列
Wq/Wk/Wv:   (d_model=4, d_k=4)     - Query/Key/Value 权重矩阵
Q,K,V:      (1, 3, 4)              - 投影后的 Query/Key/Value
Scores:     (1, 3, 3)              - 注意力分数 = Q @ K^T
Weights:    (1, 3, 3)              - softmax 后的注意力权重
Output:     (1, 3, 4)              - 输出 = Weights @ V
```

### 1. 定义输入数据 

In [1]:

import numpy as np

# 设置 NumPy 输出格式：保留4位小数，不使用科学计数法
np.set_printoptions(precision=4, suppress=True)

# ========== 1. 定义输入数据 ==========
# 玩具输入数据 (batch=1, seq_len=3, d_model=4)
# 模拟3个 token 的嵌入向量，每个 token 用4维向量表示
X = np.array([[[0.1, 0.2, 0.3, 0.4],  # Token 1 的嵌入向量
               [0.5, 0.4, 0.3, 0.2],  # Token 2 的嵌入向量
               [0.0, 0.1, 0.0, 0.1],
               [0.0, 0.1, 0.0, 0.2]]], dtype=np.float32)  # Token 4 的嵌入向量

print("X shape:", X.shape, "\nX=\n", X[0])

X shape: (1, 4, 4) 
X=
 [[0.1 0.2 0.3 0.4]
 [0.5 0.4 0.3 0.2]
 [0.  0.1 0.  0.1]
 [0.  0.1 0.  0.2]]


### 2. 定义权重矩阵

In [2]:
# ========== 2. 定义权重矩阵 ==========
# 权重矩阵 (在真实模型中通过训练学习得到)。这里我们固定数值以保证结果可复现。
# 单头：d_model=4, d_k=d_v=4，投影前后维度相同

# Query 权重矩阵：用于生成查询向量 (d_model=4, d_k=4)
Wq = np.array([[ 0.2, -0.1,  0.05,  0.0 ],
               [ 0.0,  0.1, -0.05,  0.1 ],
               [ 0.1,  0.2,  0.0,  -0.1 ],
               [-0.1,  0.0,  0.15,  0.05]], dtype=np.float32)

# Key 权重矩阵：用于生成键向量 (d_model=4, d_k=4)
Wk = np.array([[ 0.1,  0.1,  0.0,   0.05],
               [ 0.0, -0.1,  0.1,   0.0 ],
               [ 0.2,  0.0, -0.05,  0.1 ],
               [ 0.0,  0.2,  0.05, -0.1]], dtype=np.float32)

# Value 权重矩阵：用于生成值向量 (d_model=4, d_v=4)
Wv = np.array([[ 0.1,  0.0,  0.05, -0.05],
               [-0.1,  0.1,  0.0,   0.1 ],
               [ 0.2, -0.1,  0.1,   0.0 ],
               [ 0.0,  0.2, -0.1,   0.15]], dtype=np.float32)

print("Wq shape:", Wq.shape, "\nWq=\n", Wq)
print("Wk shape:", Wk.shape, "\nWk=\n", Wk)
print("Wv shape:", Wv.shape, "\nWv=\n", Wv)

Wq shape: (4, 4) 
Wq=
 [[ 0.2  -0.1   0.05  0.  ]
 [ 0.    0.1  -0.05  0.1 ]
 [ 0.1   0.2   0.   -0.1 ]
 [-0.1   0.    0.15  0.05]]
Wk shape: (4, 4) 
Wk=
 [[ 0.1   0.1   0.    0.05]
 [ 0.   -0.1   0.1   0.  ]
 [ 0.2   0.   -0.05  0.1 ]
 [ 0.    0.2   0.05 -0.1 ]]
Wv shape: (4, 4) 
Wv=
 [[ 0.1   0.    0.05 -0.05]
 [-0.1   0.1   0.    0.1 ]
 [ 0.2  -0.1   0.1   0.  ]
 [ 0.    0.2  -0.1   0.15]]


###  3. 投影到 Q, K, V

In [3]:
# ========== 3. 投影到 Q, K, V ==========
# 通过矩阵乘法将输入投影到查询、键、值空间
Q = X @ Wq  # Query:  (1, 4, 4) - 每个 token 的查询向量
K = X @ Wk  # Key:    (1, 4, 4) - 每个 token 的键向量
V = X @ Wv  # Value:  (1, 4, 4) - 每个 token 的值向量 (3,4) * (4,4) = (3,4)

print("Q shape:", Q.shape, "\nQ=\n", Q[0])
print("\nK shape:", K.shape, "\nK=\n", K[0])
print("\nV shape:", V.shape, "\nV=\n", V[0])

Q shape: (1, 4, 4) 
Q=
 [[ 0.01   0.07   0.055  0.01 ]
 [ 0.11   0.05   0.035  0.02 ]
 [-0.01   0.01   0.01   0.015]
 [-0.02   0.01   0.025  0.02 ]]

K shape: (1, 4, 4) 
K=
 [[ 0.07   0.07   0.025 -0.005]
 [ 0.11   0.05   0.035  0.035]
 [ 0.     0.01   0.015 -0.01 ]
 [ 0.     0.03   0.02  -0.02 ]]

V shape: (1, 4, 4) 
V=
 [[ 0.05   0.07  -0.005  0.075]
 [ 0.07   0.05   0.035  0.045]
 [-0.01   0.03  -0.01   0.025]
 [-0.01   0.05  -0.02   0.04 ]]


### 4. 计算注意力分数

In [4]:
# ========== 4. 计算注意力分数 ==========
# 使用缩放点积注意力：Attention(Q,K,V) = softmax(QK^T / sqrt(d_k))V
scale = 1.0 / np.sqrt(Q.shape[-1])  # 缩放因子 = 1/sqrt(d_k)，防止点积过大
attn_scores = (Q @ K.transpose(0,2,1)) * scale  # (1, 3, 3) - 每对 token 之间的相似度

print(f"缩放因子: {scale:.4f}")
print("原始注意力分数 (QK^T / sqrt(d_k)):\n", attn_scores[0])
print("attention shape: ", attn_scores.shape)
print("matrix attention score: \n", attn_scores)

缩放因子: 0.5000
原始注意力分数 (QK^T / sqrt(d_k)):
 [[ 0.0035  0.0034  0.0007  0.0015]
 [ 0.006   0.0083  0.0004  0.0009]
 [ 0.0001  0.0001  0.      0.0001]
 [-0.0001 -0.0001  0.0001  0.0002]]
attention shape:  (1, 4, 4)
matrix attention score: 
 [[[ 0.0035  0.0034  0.0007  0.0015]
  [ 0.006   0.0083  0.0004  0.0009]
  [ 0.0001  0.0001  0.      0.0001]
  [-0.0001 -0.0001  0.0001  0.0002]]]


### 5. 应用因果掩码

In [6]:
# ========== 5. 应用因果掩码 ==========
# 因果掩码：上三角区域设为 -inf，使得 softmax 后为0
# 这确保每个 token 只能关注它自己和之前的 token (自回归特性)
mask = np.triu(np.ones((1,4,4), dtype=bool), k=1)  # k=1 表示主对角线上方
print("mask shape: ", mask.shape)
print("mask matrix:\n ", mask)
attn_scores = np.where(mask, -1e9, attn_scores)

print("\n因果掩码后的注意力分数:")
print(attn_scores[0])
print("attenshion shape: ", attn_scores.shape)
print("attention matrix: \n", attn_scores)


mask shape:  (1, 4, 4)
mask matrix:
  [[[False  True  True  True]
  [False False  True  True]
  [False False False  True]
  [False False False False]]]

因果掩码后的注意力分数:
[[ 3.4625e-03 -1.0000e+09 -1.0000e+09 -1.0000e+09]
 [ 5.9875e-03  8.2625e-03 -1.0000e+09 -1.0000e+09]
 [ 8.7500e-05  1.3750e-04  5.0000e-05 -1.0000e+09]
 [-8.7500e-05 -6.2500e-05  1.3750e-04  2.0000e-04]]
attenshion shape:  (1, 4, 4)
attention matrix: 
 [[[ 3.4625e-03 -1.0000e+09 -1.0000e+09 -1.0000e+09]
  [ 5.9875e-03  8.2625e-03 -1.0000e+09 -1.0000e+09]
  [ 8.7500e-05  1.3750e-04  5.0000e-05 -1.0000e+09]
  [-8.7500e-05 -6.2500e-05  1.3750e-04  2.0000e-04]]]


### 6. 计算注意力权重

In [7]:
# ========== 6. 计算注意力权重 ==========
# 对最后一维应用 softmax，得到归一化的注意力权重
weights = np.exp(attn_scores - attn_scores.max(axis=-1, keepdims=True))  # 数值稳定性技巧
weights = weights / weights.sum(axis=-1, keepdims=True)  # 归一化


print("Weights shape:", weights.shape, "\nAttention Weights (causal)=\n", weights[0])
print("\n注意：每行和为1，且上三角为0 (因果掩码)")
print("weights shape: ", weights.shape)
print("weights matrix: ", weights)

Weights shape: (1, 4, 4) 
Attention Weights (causal)=
 [[1.     0.     0.     0.    ]
 [0.4994 0.5006 0.     0.    ]
 [0.3333 0.3333 0.3333 0.    ]
 [0.25   0.25   0.25   0.25  ]]

注意：每行和为1，且上三角为0 (因果掩码)
weights shape:  (1, 4, 4)
weights matrix:  [[[1.     0.     0.     0.    ]
  [0.4994 0.5006 0.     0.    ]
  [0.3333 0.3333 0.3333 0.    ]
  [0.25   0.25   0.25   0.25  ]]]


### 7. 计算输出

In [8]:
# ========== 7. 计算输出 ==========
# 用注意力权重对值向量进行加权求和
# 每个 token 的输出是所有 token 值向量的加权组合
out = weights @ V  # (1, 3, 4) - 最终的自注意力输出


print("Output shape:", out.shape, "\nOutput=\n", out[0])

Output shape: (1, 4, 4) 
Output=
 [[ 0.05    0.07   -0.005   0.075 ]
 [ 0.06    0.06    0.015   0.06  ]
 [ 0.0367  0.05    0.0067  0.0483]
 [ 0.025   0.05   -0.      0.0462]]


In [9]:
# ========== 8. 输出投影 (完整 Transformer 所需) ==========
# 在完整的 Transformer 中，输出投影 W_o 混合各通道信息
# 单头且 d_v=d_model=4 时，out 已是 (1,3,4)，W_o 做通道混合而非升维

# 输出投影矩阵：(d_v=4, d_model=4)
W_o = np.array([[ 0.3,  0.1, -0.2,  0.4],
                [-0.1,  0.2,  0.3, -0.1],
                [ 0.2, -0.1,  0.1,  0.2],
                [ 0.0,  0.15, -0.05, 0.3]], dtype=np.float32)

print(f"输出投影矩阵 W_o 的形状: {W_o.shape} - (d_v=4, d_model=4)")

# 执行输出投影
final_out = out @ W_o  # (1, 3, 4) @ (4, 4) = (1, 3, 4)

print(f"\n最终输出形状: {final_out.shape} - 与输入 X 的形状一致！")
print("Final Output=\n", final_out[0])

输出投影矩阵 W_o 的形状: (4, 4) - (d_v=4, d_model=4)

最终输出形状: (1, 4, 4) - 与输入 X 的形状一致！
Final Output=
 [[0.007  0.0308 0.0067 0.0345]
 [0.015  0.0255 0.0045 0.039 ]
 [0.0073 0.0203 0.0059 0.0255]
 [0.0025 0.0194 0.0077 0.0189]]
